# RAG Generation Notebook — Gemini

End-to-end pipeline: query → vector retrieval → Gemini generation.

| Step | Description |
|---|---|
| 1 | Install dependencies |
| 2 | Configuration (model, retrieval, persona) |
| 3 | Build retriever + resolve chunks from S3 |
| 4 | Run retrieval |
| 5 | Build combined prompt |
| 6 | Gemini inference |
| 7 | Save generation record (optional) |

## 1. Dependencies (run once per environment)

In [1]:
%pip install sentence-transformers boto3 pandas google-genai python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuration

Edit the cells below to change the query, persona, or retrieval settings.

`GEMINI_API_KEY` and **`GEMINI_MODEL`** must be set in the project root **`.env`** file (loaded automatically). You can still override any variable by exporting it in your shell before starting Jupyter.

In [30]:
import os
from pathlib import Path

from dotenv import load_dotenv


def _repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    return Path.cwd()


load_dotenv(_repo_root() / ".env", override=False)

# ── Student query ────────────────────────────────────────────────────────────
USER_QUERY = "How do I express polite requests in Japanese?"

# ── Gemini — from .env (or shell): GEMINI_API_KEY, GEMINI_MODEL ──────────────
GEMINI_MODEL   = os.environ.get("GEMINI_MODEL", "").strip()
MAX_TOKENS     = int(os.environ.get("GENERATION_MAX_TOKENS", "2048"))
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()

if not GEMINI_MODEL:
    raise EnvironmentError(
        "GEMINI_MODEL is not set. Add it to the project root .env file "
        '(e.g. GEMINI_MODEL=gemini-2.5-flash) or export it in your shell.'
    )
if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY is not set. Add it to the project root .env file "
        "or export it in your shell before running."
    )

# ── Tutor persona ─────────────────────────────────────────────────────────────
TUTOR_PERSONA = (
    "You are an encouraging and knowledgeable Japanese language tutor "
    "specialising in domain-specific translation. "
    "You explain grammar and vocabulary clearly, give concrete examples, "
    "and always keep answers practical and supportive. "
    "When the student asks about translation, prefer the most natural "
    "Japanese phrasing and point out any register or politeness considerations."
)

# ── Retrieval settings ────────────────────────────────────────────────────────
TOP_K               = 5        # final chunks passed to the prompt
VECTOR_QUERY_TOP_K  = 100      # wider vector search before line-band filter
MAX_SOURCE_LINE     = 100      # keep only chunks from the first N JSONL lines

VECTOR_BUCKET = os.environ.get("RAG_VECTOR_BUCKET", "is469-genai-grp-project")
INDEX_NAME    = os.environ.get("RAG_VECTOR_INDEX",  "rag-vector-2")
REGION        = os.environ.get("VECTORS_AWS_DEFAULT_REGION", "ap-southeast-1")

_kb_env = os.environ.get("RAG_KB_DIR", "").strip()

print(f"Query  : {USER_QUERY}")
print(f"Model  : {GEMINI_MODEL}")
print(f"Bucket : {VECTOR_BUCKET}  |  Index: {INDEX_NAME}  |  Region: {REGION}")

Query  : How do I express polite requests in Japanese?
Model  : gemini-2.5-flash
Bucket : is469-genai-grp-project  |  Index: rag-vector-2  |  Region: ap-southeast-1


## 3. Build retriever

In [24]:
import sys

def project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    raise RuntimeError("Could not find project root (folder with src/retrieval). cd to the repo root or rag/.")

_ROOT = project_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

KB_DIR = Path(_kb_env) if _kb_env else (_ROOT / "kb")

from src.retrieval.s3_vectors_rag import S3VectorsRAGRetriever
import src.retrieval.s3_vectors_rag as _s3_rag

# Patch kb path guesser to match this repo's file naming
def _guess_kb_paths(kb_dir: Path, source_file: str) -> list[Path]:
    raw = (source_file or "").strip()
    if not raw:
        return []
    name = Path(raw).name
    if name.endswith(".jsonl"):
        basename = name
        stem_no_ext = Path(name).stem
    else:
        stem_no_ext = name
        basename = f"{name}.jsonl"
    candidates: list[Path] = []
    seen: set[Path] = set()
    def add(p: Path) -> None:
        if p not in seen:
            seen.add(p)
            candidates.append(p)
    add(kb_dir / basename)
    add(kb_dir / f"{stem_no_ext}_vectors.jsonl")
    if stem_no_ext.endswith("_embedded_full"):
        add(kb_dir / f"{stem_no_ext[:-len('_full')]}.jsonl")
    if stem_no_ext.endswith("_embedded") and not stem_no_ext.endswith("_embedded_full"):
        add(kb_dir / f"{stem_no_ext}_full.jsonl")
    return [p for p in candidates if p.is_file()]

_s3_rag._guess_kb_paths = _guess_kb_paths

retriever = S3VectorsRAGRetriever(
    vector_bucket_name=VECTOR_BUCKET,
    index_name=INDEX_NAME,
    region_name=REGION,
    kb_dir=KB_DIR,
    top_k=VECTOR_QUERY_TOP_K,
)
print(f"Project root : {_ROOT}")
print(f"KB dir       : {KB_DIR} (exists={KB_DIR.is_dir()})")
print("Retriever ready (encoder loads lazily on first retrieve).")

Project root : c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469
KB dir       : c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\kb (exists=True)
Retriever ready (encoder loads lazily on first retrieve).


In [25]:
from src.utils.aws_profiles import s3vectors_client

vectors_client = s3vectors_client(region_name=REGION)

def resolve_chunks_from_s3(keys: list[str]) -> dict[str, str]:
    """Fetch chunk text from rag-vector-2 metadata for unresolved keys."""
    if not keys:
        return {}
    response = vectors_client.get_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=INDEX_NAME,
        keys=keys,
        returnData=False,
        returnMetadata=True,
    )
    result = {}
    for vec in response.get("vectors", []):
        key  = vec.get("key")
        meta = vec.get("metadata") or {}
        text = meta.get("text") or meta.get("chunk_text") or meta.get("content")
        if key and text:
            result[key] = text
    return result

print("✓ s3vectors_client ready.")

✓ s3vectors_client ready.


## 4. Run retrieval

In [26]:
import pandas as pd

_, all_chunks = retriever.retrieve(USER_QUERY)

def _dist_key(c):
    return float(c.distance) if c.distance is not None else float("inf")

_in_band = [c for c in all_chunks if 1 <= c.source_line <= MAX_SOURCE_LINE]
_in_band.sort(key=_dist_key)
chunks = _in_band[:TOP_K]

print(
    f"Vector pool : {len(all_chunks)} hits (top {VECTOR_QUERY_TOP_K}); "
    f"in source lines 1–{MAX_SOURCE_LINE}: {len(_in_band)}; using {len(chunks)}."
)

# Resolve any unresolved chunks from S3 metadata
unresolved_keys = [c.key for c in chunks if (c.text or "").startswith("(no local text resolved")]
if unresolved_keys:
    print(f"Resolving {len(unresolved_keys)} chunk(s) from S3...")
    s3_map = resolve_chunks_from_s3(unresolved_keys)
    for chunk in chunks:
        if (chunk.text or "").startswith("(no local text resolved"):
            chunk.text = s3_map.get(chunk.key, chunk.text)

# Build formatted context string
context = "\n---\n".join(
    f"[{c.source_file} L{c.source_line}]\n{c.text or '(unresolved)'}"
    for c in chunks
)

# Summary DataFrame
_preview = 320
hits_df = pd.DataFrame([
    {
        "rank": i,
        "distance": round(float(c.distance), 6) if c.distance is not None else None,
        "key": c.key,
        "source_file": c.source_file,
        "source_line": c.source_line,
        "resolved": not (c.text or "").startswith("(no local text resolved"),
        "text_preview": (c.text or "")[:_preview],
        "chunk_text": c.text or "",
    }
    for i, c in enumerate(chunks, start=1)
])

print("\n=== Retrieval hits ===")
hits_df[["rank", "distance", "source_file", "source_line", "resolved", "text_preview"]]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector pool : 94 hits (top 100); in source lines 1–100: 3; using 3.

=== Retrieval hits ===


,rank,distance,source_file,source_line,resolved,text_preview
0,1,0.109273,eng_jap_chunks_embedded_full.jsonl,79,True,[1] EN: I'm tired.\n[1] JA: 私は、疲れています。\n\n[2] ...
1,2,0.119010,eng_jap_chunks_embedded_full.jsonl,82,True,[1] EN: I must admit that I snore.\n[1] JA: いび...
2,3,0.119370,eng_jap_chunks_embedded_full.jsonl,80,True,"[1] EN: The check, please.\n[1] JA: 勘定を頼むよ。\n\..."


## 5. Build combined prompt

The system instruction holds the **tutor persona**.  
The user turn holds the **retrieved reference material** + the **student question**.

In [31]:
def build_user_message(query: str, retrieved_context: str) -> str:
    if retrieved_context.strip():
        ctx_block = (
            "### Retrieved reference material\n"
            "The following excerpts come from the project knowledge base "
            "(glossary, translation memory, grammar notes, style guide). "
            "Use them to ground your answer.\n\n"
            f"{retrieved_context}\n\n"
            "---\n"
        )
    else:
        ctx_block = ""  # graceful fallback when retrieval returns nothing
    return ctx_block + f"### Student question\n{query}"


user_message = build_user_message(USER_QUERY, context)

print("=== System instruction (persona) ===")
print(TUTOR_PERSONA)
print("\n=== User message (first 1200 chars) ===")
print(user_message[:1200], "..." if len(user_message) > 1200 else "")

=== System instruction (persona) ===
You are an encouraging and knowledgeable Japanese language tutor specialising in domain-specific translation. You explain grammar and vocabulary clearly, give concrete examples, and always keep answers practical and supportive. When the student asks about translation, prefer the most natural Japanese phrasing and point out any register or politeness considerations.

=== User message (first 1200 chars) ===
### Retrieved reference material
The following excerpts come from the project knowledge base (glossary, translation memory, grammar notes, style guide). Use them to ground your answer.

[eng_jap_chunks_embedded_full.jsonl L79]
[1] EN: I'm tired.
[1] JA: 私は、疲れています。

[2] EN: Who wants some hot chocolate?
[2] JA: ホットチョコレート欲しい人ー？

[3] EN: Speak more slowly, please!
[3] JA: もっとゆっくり話してください！

[4] EN: When do we arrive?
[4] JA: いつ着くの？

[5] EN: The check, please.
[5] JA: 勘定お願いします。

[6] EN: The check, please.
[6] JA: 勘定書をお願いします。

[7] EN: The check, please.
[

## 6. Gemini inference

Uses the `google-genai` SDK (`google.genai`).  
The persona is passed as `system_instruction`; the combined context + query is the user turn.

In [34]:
from google import genai
from google.genai import types
from IPython.display import Markdown, display

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=user_message,
    config=types.GenerateContentConfig(
        system_instruction=TUTOR_PERSONA,
        max_output_tokens=MAX_TOKENS,
    ),
)

response_text = response.text

# Usage metadata (not all models populate every field)
usage = response.usage_metadata
print(f"Model          : {GEMINI_MODEL}")
print(f"Input tokens   : {getattr(usage, 'prompt_token_count', 'n/a')}")
print(f"Output tokens  : {getattr(usage, 'candidates_token_count', 'n/a')}")
print("\n" + "="*60)
print("TUTOR RESPONSE")
print("="*60 + "\n")

try:
    display(Markdown(response_text))
except Exception:
    print(response_text)

Model          : gemini-2.5-flash
Input tokens   : 748
Output tokens  : 908

TUTOR RESPONSE



That's a fantastic question, and one of the most practical things to learn when starting out in Japanese! Expressing polite requests is essential for smooth communication.

Let's break down how to do this, looking at your reference material and adding some more common patterns.

---

### The Most Common & Versatile Way: 「～てください」

This is your go-to form for making polite requests. It's formed by taking the **te-form (て形)** of a verb and adding **ください (kudasai)**.

*   **Formation:** Verb (て形) + ください
*   **Meaning:** "Please do [verb]"

**Example from your reference:**

*   **EN:** Speak more slowly, please!
*   **JA:** もっとゆっくり**話してください**！ (motto yukkuri **hanashite kudasai**!)
    *   Here, 話す (hanasu - to speak) becomes 話して (hanashite) in its te-form, then ください is added.

**Other common examples:**

*   座る (suwaru - to sit) → **座ってください** (suwatte kudasai) - Please sit down.
*   待つ (matsu - to wait) → **待ってください** (matte kudasai) - Please wait.
*   書く (kaku - to write) → **書いてください** (kaite kudasai) - Please write it.

This form is widely applicable and generally polite enough for most everyday situations, like asking a shop assistant, a classmate, or someone you're speaking to for the first time.

---

### Requesting Items or Services: 「～をお願いします」 or 「～ください」

When you want to ask for a *thing* or a *service*, 「お願いします (onegai shimasu)」 is incredibly useful. 「お願いします」 literally means "I humbly request" or "I ask of you."

*   **Formation 1:** Noun + を (o) + お願いします
*   **Formation 2:** Noun + お願いします (often used informally, or when "を" is implied)
*   **Meaning:** "Please give me [noun]" or "I'd like [noun]"

**Examples from your reference:**

*   **EN:** The check, please.
*   **JA:** 勘定**お願いします**。 (kanjou **onegai shimasu**.)
    *   Here, 勘定 (kanjou - the check/bill) is followed by お願いします.
*   **JA:** 勘定書**をお願いします**。 (kanjousho **wo onegai shimasu**.)
    *   Same meaning, just with the noun 勘定書 (kanjousho - check/bill paper) and the particle を.
*   **JA:** お愛想**お願いします**。 (oaiso **onegai shimasu**.)
    *   お愛想 (oaiso) is another common way to say "the check" in a restaurant, followed by お願いします.

You can also combine a noun directly with **ください** in these contexts, especially for common items:

*   **Formation:** Noun + ください
*   **Meaning:** "Please give me [noun]"

**Example:**

*   お水**ください** (omizu **kudasai**) - Please give me water.
*   コーヒー**ください** (koohii **kudasai**) - Please give me coffee.

This form is direct but generally polite enough when ordering in a casual setting.

---

### More Polite & Indirect Requests: 「～ていただけますか / いただけませんか」

These forms are more formal and indirect, making them sound even more polite. They literally mean "Would you be able to do [verb] for me?" or "Wouldn't you be able to do [verb] for me?"

*   **Formation:** Verb (て形) + いただけますか (itadakemasu ka)
*   **Formation:** Verb (て形) + いただけませんか (itadakemasen ka)
*   **Meaning:** "Could you please [verb] for me?" / "Would